# data

In [ ]:
# Cell 1: data
import pandas as pd
import numpy as np
import re
import os

elements = [
    'Al', 'Ag', 'Ba', 'K', 'Se', 'Sn', 'C', 'V', 'Au', 'Mn', 'Cd', 'Ge', 'Mo', 
    'Pb', 'Na', 'Rb', 'Zn', 'Ti', 'Ta', 'Cs', 'Pt', 'In', 'Ca', 'Mg', 'Sb', 
    'Zr', 'W', 'Hf', 'Ga', 'P', 'Cr', 'Cu', 'Ni', 'Fe', 'Co', 'Nb', 'Li', 'Si', 'Bi'
]

file_path = 'surface tension data.csv'

try:
    df = pd.read_csv(file_path, encoding='utf-8')
    
except UnicodeDecodeError:
    try:
        df = pd.read_csv(file_path, encoding='gbk')
    
    except UnicodeDecodeError:
        df = pd.read_csv(file_path, encoding='latin1')
    

for el in elements:
    df[el] = 0.0

def parse_composition(comp_str):
    comp_dict = {}
    
    matches = re.findall(r'([A-Z][a-z]*)(\d+\.?\d*)', str(comp_str))
    for el, frac in matches:
        if el in elements:
            comp_dict[el] = float(frac)
    return comp_dict


for index, row in df.iterrows():
    comp_dict = parse_composition(row['Composition'])
    for el, frac in comp_dict.items():
        df.at[index, el] = frac


initial_len = len(df)

df = df.dropna(subset=['Composition', 'Surface Tension', 'T'])
df = df.drop_duplicates()
cleaned_len = len(df)

df_cleaned = df.copy()
display(df_cleaned.head())

正在加载数据...
成功使用 utf-8 编码读取文件。
正在解析合金成分...
数据清洗完成。原始数据行数: 1，清洗后行数: 1


,Composition,Surface Tension,T,Al,Ag,Ba,K,Se,Sn,C,...,P,Cr,Cu,Ni,Fe,Co,Nb,Li,Si,Bi
0,Ag0.5-Sn0.5,100,1000,0.0,0.5,0.0,0.0,0.0,0.5,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


# feature construction

In [ ]:
# Cell 2: feature construction
from matminer.featurizers.conversions import StrToComposition
from matminer.featurizers.composition import ElementProperty
import warnings

warnings.filterwarnings('ignore')

df_features = df_cleaned.copy()
df_features['clean_comp'] = df_features['Composition'].str.replace('-', '')

stc = StrToComposition(target_col_id='composition_obj', overwrite_data=True)
df_features = stc.featurize_dataframe(df_features, 'clean_comp')

ep = ElementProperty.from_preset(preset_name="magpie")
df_features = ep.featurize_dataframe(df_features, col_id='composition_obj')

required_descriptors = [
    "MagpieData mean AtomicWeight", 
    "MagpieData range CovalentRadius", 
    "MagpieData mean MeltingT", 
    "MagpieData mean Electronegativity", 
    "MagpieData avg_dev NValence" 
]


target_p_valence_col = "MagpieData mean NpValence" 
if target_p_valence_col in df_features.columns:
    df_features.rename(columns={target_p_valence_col: 'avg p valence electrons'}, inplace=True)
elif 'avg p valence electrons' not in df_features.columns:
    
    print("plese check Matminer ed")
    df_features['avg p valence electrons'] = 0 


feature_cols = ['T'] + elements + [
    "MagpieData mean AtomicWeight", 
    "MagpieData range CovalentRadius", 
    "MagpieData mean MeltingT", 
    "MagpieData mean Electronegativity", 
    "avg p valence electrons"
]


for col in feature_cols:
    if col not in df_features.columns:
        df_features[col] = 0.0

df_final = df_features[['Composition', 'Surface Tension'] + feature_cols]

display(df_final.head())

正在生成 Matminer 特征...


ElementProperty: 100%|██████████| 1/1 [00:00<00:00,  6.50it/s]


特征生成完成！


,Composition,Surface Tension,T,Al,Ag,Ba,K,Se,Sn,C,...,Co,Nb,Li,Si,Bi,MagpieData mean AtomicWeight,MagpieData range CovalentRadius,MagpieData mean MeltingT,MagpieData mean Electronegativity,avg p valence electrons
0,Ag0.5-Sn0.5,100,1000,0.0,0.5,0.0,0.0,0.0,0.5,0.0,...,0.0,0.0,0.0,0.0,0.0,113.2891,6.0,870.005,1.945,1.0


# model prediction

In [ ]:
# Cell 3: prediction
import joblib
import os
from sklearn.preprocessing import StandardScaler

model_path = 'general_model.pkl'

try:

    elements_order = [
        'Al', 'Ag', 'Ba', 'K', 'Se', 'Sn', 'C', 'V', 'Au', 'Mn', 'Cd', 'Ge', 'Mo', 
        'Pb', 'Na', 'Rb', 'Zn', 'Ti', 'Ta', 'Cs', 'Pt', 'In', 'Ca', 'Mg', 'Sb', 
        'Zr', 'W', 'Hf', 'Ga', 'P', 'Cr', 'Cu', 'Ni', 'Fe', 'Co', 'Nb', 'Li', 'Si', 'Bi'
    ]
    descriptors_order = [
        "MagpieData mean AtomicWeight", 
        "MagpieData range CovalentRadius", 
        "MagpieData mean MeltingT", 
        "MagpieData mean Electronegativity", 
        "avg p valence electrons"
    ]
    
    expected_cols = ['T'] + elements_order + descriptors_order
    

    for col in expected_cols:
        if col not in df_final.columns:
            df_final[col] = 0.0
            

    X_predict = df_final[expected_cols]
    

    scaler = StandardScaler()
    X_predict_scaled = scaler.fit_transform(X_predict)


    model = joblib.load(model_path)
    predictions = model.predict(X_predict_scaled)
    

    df_final['Predicted Surface Tension'] = predictions
    

    result_df = df_final[['Composition', 'T', 'Surface Tension', 'Predicted Surface Tension']]

    output_file = 'predicted_results.csv'
    result_df.to_csv(output_file, index=False, encoding='utf-8-sig')
    print(f"\nsaved to: {output_file}")

except FileNotFoundError:
    print(f"can not find {model_path}")
except Exception as e:
    print(f"error: {e}")

正在准备数据并加载模型进行预测...

预测完成！

所有预测结果已成功保存至: predicted_results.csv
